In [1]:
from modules.dataset.make_dataset import make_dataset, dataset_split, NpzDataPipeline
from modules.nn import BranchedCoughCNN, DefaultCNN
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from modules.preprocessing.dataframe_process import filter_coughvid_df

In [2]:
def make_path(audio_name: str) -> str:
    return f"data/public_dataset_v3_wav/{audio_name.split('.')[0]}.wav"

df: pl.DataFrame = filter_coughvid_df(
    pl.read_csv(
        'data/tabular_form/tabular_form/coughvid_v3.csv', 
        schema_overrides={"age":pl.Float32}
    )
)

# point audio_name at the actual wav files on disk
df = df.with_columns(pl.col("audio_name").map_elements(make_path, return_dtype=pl.String))


In [3]:
# generates dataset

make_dataset(df, 16_000, out="data\\final_processed_dataset.npz")


processing audio:   0%|          | 0/12885 [00:00<?, ?it/s]

In [ ]:
dataset: NpzDataPipeline = NpzDataPipeline("data\final_processed_dataset.npz")
dataset.load_data()
#train_dataset, val_dataset, test_dataset = dataset_split(dataset)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, 64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_dataset, 64, shuffle=True, num_workers=4, pin_memory=True)

In [ ]:
model: torch.nn.Module | None = None

In [ ]:
device: torch.types.Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


In [ ]:
loader

In [ ]:
for epoch in range(num_epochs):
    # Training Phase
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        # Move inputs to GPU. non_blocking=True works with pin_memory=True in DataLoader
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) # Memory Speedup
        
        # Runs the forward pass in mixed precision (float16/float32 mixed)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Scale the loss and do the backward pass
        scaler.scale(loss).backward()
        
        # Step the optimizer through the scaler (unscales gradients behind the scenes)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
    # Step the learning rate scheduler at the end of the epoch
    scheduler.step()
    
    # 4. Optimized Validation Phase (Inference)
    optimized_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    # inference_mode() is faster and uses less memory than no_grad()
    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    # Log metrics
    train_epoch_loss = running_loss / len(train_loader)
    val_epoch_loss = val_loss / len(val_loader)
    val_accuracy = 100.0 * correct / total
    
    print(f"Epoch [{epoch+1}/{num_epochs}] -> "
          f"Train Loss: {train_epoch_loss:.4f} | "
          f"Val Loss: {val_epoch_loss:.4f} | "
          f"Val Acc: {val_accuracy:.2f}%")

In [ ]:
# ---- Experiment setup (shared) ----
from modules.dataset.make_dataset import AugmentedSubset
from modules.preprocessing.audio_process import WaveformAugmentor, SpecAugmentations, MelSpectrogramPipeline, SpectMFCCPipeline
from tqdm.auto import tqdm

num_epochs: int = 30
SEED: int = 42  # identical split across all 3 experiments

# ponytail: fill these manually (healthy, COVID-19, symptomatic)
CLASS_WEIGHTS: torch.Tensor = torch.tensor([1.0, 1.0, 1.0], device=device)
criterion: nn.Module = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

mel_train_plain: SpectMFCCPipeline = SpectMFCCPipeline().to(device)  # no spec aug
mel_train_spec: SpectMFCCPipeline = SpectMFCCPipeline(transforms=SpecAugmentations()).to(device)
mel_eval: SpectMFCCPipeline = SpectMFCCPipeline().to(device)         # plain for validation

# speed: compile only the eval mel path; train paths stay eager (spec-aug masks/augmentor randomize per step)
mel_eval = torch.compile(mel_eval) if hasattr(torch, "compile") else mel_eval

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

In [ ]:
# ---- Seeded split (shared across experiments) ----
dataset.load_data()

train_subset, val_subset, _test_subset = dataset_split(dataset, seed=SEED)

In [ ]:
# ---- Experiment runner (based on the base training loop) ----
from torch.utils.data import Subset

def run_experiment(name: str, use_wave: bool, use_spec: bool) -> dict[str, list[float]]:
    history: dict[str, list[float]] = {"train_loss": [], "val_loss": [], "val_acc": []}

    # Fresh model/optimizer per experiment so runs are independent
    model: nn.Module  = torch.compile(DefaultCNN(num_classes=3)).to(device)  # speed: compiled model
    optimizer: optim.Adam = optim.Adam(model.parameters(), lr=1e-4)
    scaler: torch.amp.GradScaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler: optim.lr_scheduler.CosineAnnealingLR = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    mel_train: SpectMFCCPipeline = mel_train_spec if use_spec else mel_train_plain

    # Train loader: AugmentedSubset (WaveformAugmentor) when use_wave, else plain Subset
    train_ds: AugmentedSubset | Subset = AugmentedSubset(
        dataset, train_subset.indices, 
        WaveformAugmentor(16_000)
    ) if use_wave else Subset(dataset, train_subset.indices)
    
    train_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        train_ds, 64, shuffle=True, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,  # speed: no worker respawn / keep queues full
    )
    val_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        Subset(dataset, val_dataset.indices), 64, shuffle=False, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
  
        for waveforms, labels in tqdm(train_loader, desc=f"[{name}] train {epoch+1}/{num_epochs}", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                images = mel_train(waveforms).unsqueeze(1)  # (B, 1, F, T) for Conv2d
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.inference_mode():
            for waveforms, labels in tqdm(val_loader, desc=f"[{name}] val {epoch+1}/{num_epochs}", leave=False):
                waveforms = waveforms.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    images = mel_eval(waveforms).unsqueeze(1)
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        train_epoch_loss = running_loss / len(train_loader)
        val_epoch_loss = val_loss / len(val_loader)
        val_accuracy = 100.0 * correct / total

        history["train_loss"].append(train_epoch_loss)
        history["val_loss"].append(val_epoch_loss)
        history["val_acc"].append(val_accuracy)

        print(f"[{name}] Epoch [{epoch+1}/{num_epochs}] -> "
              f"Train Loss: {train_epoch_loss:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} | "
              f"Val Acc: {val_accuracy:.2f}%")

    return history

In [ ]:
# ---- Run all 3 experiments ----
results: dict[str, dict[str, list[float]]] = {}

results["both"] = run_experiment("both (wave + spec)", use_wave=True, use_spec=True)
results["wave_only"] = run_experiment("wave only", use_wave=True, use_spec=False)
results["spec_only"] = run_experiment("spec only", use_wave=False, use_spec=False)

# Comparison table
print("\n=== Experiment Comparison ===")
print(f"{'Experiment':<20} {'Best Val Acc':>12} {'Min Val Loss':>13}")
for name, hist in results.items():
    best_acc: float = max(hist["val_acc"])
    min_loss: float = min(hist["val_loss"])
    print(f"{name:<20} {best_acc:>11.2f}% {min_loss:>13.4f}")